# skyhunt — uruchomienie na Colab (GPU)

1. Wgraj nagrania do `MyDrive/skyhunt/raw/` (np. `DSCF4641.MOV`, `dark_frames.MOV`).
2. Uruchom komórki od góry.
3. Po rozłączeniu Colaba uruchom wszystko jeszcze raz: gotowe etapy są pomijane (`manifest.json` per plik na Drive).

Własny config: skopiuj `config.yaml` do `MyDrive/skyhunt/config.yaml`; ma pierwszeństwo przed wersją z repo.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = 'https://github.com/LukaszCiesielski-lodz/skytrack_uap.git'
BRANCH = 'main'
REPO = '/content/skytrack_uap'
DRIVE_ROOT = '/content/drive/MyDrive/skyhunt'
RAW = f'{DRIVE_ROOT}/raw'
OUT = f'{DRIVE_ROOT}/out'
CONFIG = f'{DRIVE_ROOT}/config.yaml' if os.path.exists(f'{DRIVE_ROOT}/config.yaml') else f'{REPO}/config.yaml'
os.makedirs(RAW, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print('config:', CONFIG)
print('nagrania:', sorted(os.listdir(RAW)))

In [ ]:
if os.path.isdir(f'{REPO}/.git'):
    !git -C {REPO} fetch -q origin && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} reset -q --hard origin/{BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO}
!git -C {REPO} log -1 --oneline

In [ ]:
!pip install -q -e "{REPO}[gpu,dev]"
!pip install -q PyNVVideoCodec || echo 'PyNVVideoCodec: instalacja nieudana (bench pokaże błąd)'
# torchcodec musi pasować do zainstalowanego torch: --no-deps, żeby nie podmienił torch Colaba.
import torch
CU = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
!pip install -q --no-deps torchcodec --index-url https://download.pytorch.org/whl/{CU} || echo 'torchcodec: niedostępny'
print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'GPU', torch.cuda.is_available())

## Testy
Testy `video` używają prawdziwego nagrania (parytet backendów bit w bit na 48 klatkach).

In [ ]:
TEST_VIDEO = f'{RAW}/DSCF4641.MOV'
os.environ['SKYHUNT_TEST_VIDEO'] = TEST_VIDEO if os.path.exists(TEST_VIDEO) else ''
!cd {REPO} && python -m pytest -q -rs

## Metadane i przepustowość dekodowania
Wynik benchmarku (`decode_bench.md`) wklej do README albo prześlij w czacie.

In [ ]:
!skyhunt probe "{RAW}" --config "{CONFIG}"

In [ ]:
!skyhunt bench-decode "{RAW}/DSCF4641.MOV" --config "{CONFIG}" --frames 480 --out "{OUT}/_bench"

## Pipeline
M0: `probe` (metadane, wstępny czas) i `stack` (średnia, max, statystyki per klatka).

In [ ]:
!skyhunt run "{RAW}" --config "{CONFIG}" --out "{OUT}"

In [ ]:
import glob
from IPython.display import Image, display

!skyhunt status "{RAW}" --config "{CONFIG}" --out "{OUT}"
for p in sorted(glob.glob(f'{OUT}/*/stack_max_minus_mean.png')):
    print(p)
    display(Image(p, width=1200))